# Differentiation Analysis — How the ML System and Controller Distinguish Cases

**No training. Loads pretrained checkpoints and produces differentiation-focused analysis figures/tables.**

Required Kaggle dataset inputs:
- `wound-datasets` (test images + masks)
- `polar-v2-weights` (contains `best.pth`)
- `detr-extended-weights` (contains `best.pth`)
- `autoregressive-extended-weights` (contains `best.pth`)

This notebook helps answer: *How does the system differentiate one case from another?*

In [1]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
assert torch.cuda.is_available(), 'No GPU! Select T4 x2 in Kaggle Settings.'
device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)} | PyTorch: {torch.__version__}')

IMAGE_SIZE = 256
NUM_RADII_64 = 64
NUM_RADII_128 = 128
D_MODEL = 256
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
MEAN_T = torch.tensor(MEAN).view(3,1,1)
STD_T  = torch.tensor(STD).view(3,1,1)

Path('figures').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)
print('Setup complete.')

GPU: Tesla T4 | PyTorch: 2.10.0+cu128
Setup complete.


## 1) Load test images (same pattern as Notebook 03)

In [2]:
_candidates = [
    Path('/kaggle/input/datasets/dianisay/wound-datasets/Datasets'),
    Path('/kaggle/input/wound-datasets/Datasets'),
    Path('/kaggle/input/datasets/dianisay/wound-datasets'),
    Path('/kaggle/input/wound-datasets'),
]
BASE_PATH = next((p for p in _candidates if p.exists()), _candidates[0])
print(f'BASE_PATH: {BASE_PATH} (exists={BASE_PATH.exists()})')

WOUND_DATASETS = [
    {'name': 'Medetec', 'root': BASE_PATH / 'Medetec_foot_ulcer_224',
     'splits': {'test': ('test/images', 'test/labels')}},
    {'name': 'AZH', 'root': BASE_PATH / 'wound_dataset' / 'azh_wound_care_center_dataset_patches',
     'splits': {'test': ('test/images', 'test/labels')}},
]

# Slightly larger sample than notebook 03 for analysis robustness
IMAGES_PER_DATASET = 12
fixed_images = []
for ds in WOUND_DATASETS:
    root = ds['root']
    if not root.exists():
        print(f'  {ds["name"]}: NOT FOUND at {root}')
        continue

    img_rel, lbl_rel = ds['splits']['test']
    img_dir = root / img_rel
    lbl_dir = root / lbl_rel
    if not img_dir.exists() or not lbl_dir.exists():
        print(f'  {ds["name"]}: dirs not found')
        continue

    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    lbl_files = {p.stem.lower(): p for p in lbl_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}

    count = 0
    for img_p in imgs:
        if count >= IMAGES_PER_DATASET:
            break
        mask_p = lbl_files.get(img_p.stem.lower())
        if not mask_p:
            continue

        raw_mask = np.array(Image.open(mask_p).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST))
        if raw_mask.max() <= 1:
            raw_mask = raw_mask * 255
        gt_bin = (raw_mask > 127).astype(np.uint8)
        if gt_bin.sum() < 50:
            continue

        img_pil = Image.open(img_p).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        img_np = np.array(img_pil)
        img_t = torch.from_numpy(img_np.copy()).float().permute(2,0,1) / 255.0
        img_t = T.Normalize(MEAN, STD)(img_t)

        fixed_images.append({
            'ds_name': ds['name'],
            'image': img_t,
            'raw_mask': gt_bin,
            'filename': img_p.name,
            'image_path': str(img_p),
            'mask_path': str(mask_p),
        })
        count += 1

    print(f'  {ds["name"]}: {count} images selected')

print(f'\nTotal fixed images: {len(fixed_images)}')
print(f'Datasets in sample: {sorted(set(f["ds_name"] for f in fixed_images))}')

BASE_PATH: /kaggle/input/datasets/dianisay/wound-datasets/Datasets (exists=True)
  Medetec: 8 images selected
  AZH: 12 images selected

Total fixed images: 20
Datasets in sample: ['AZH', 'Medetec']


## 2) Model architectures (inference-only, compatible with Notebook 03 checkpoints)

In [4]:
# Shared encoder
class CNNTransformerEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL, nhead=8, num_layers=2):
        super().__init__()
        backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        )
        self.proj = nn.Conv2d(512, d_model, kernel_size=1)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x):
        feat = self.stem(x)              # [B, 512, 8, 8] for 256x256
        feat = self.proj(feat)           # [B, D, 8, 8]
        B, D, H, W = feat.shape
        tokens = feat.flatten(2).transpose(1,2)  # [B, 64, D]
        mem = self.encoder(tokens)
        return mem

# Polar v2 decoder model (predicts polygon points)
class PolarModelV2(nn.Module):
    def __init__(self, d_model=D_MODEL, num_radii=NUM_RADII_128):
        super().__init__()
        self.encoder = CNNTransformerEncoder(d_model=d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2*num_radii)
        )
        self.num_radii = num_radii

    def forward(self, x):
        mem = self.encoder(x)                 # [B, 64, D]
        pooled = mem.mean(dim=1)             # [B, D]
        out = self.head(pooled).view(-1, self.num_radii, 2)
        pts = torch.sigmoid(out)             # normalized xy
        return {'points': pts, 'memory': mem}

# DETR-like point decoder
class DETRPointDecoder(nn.Module):
    def __init__(self, d_model=D_MODEL, num_points=NUM_RADII_64, nhead=8, num_layers=2):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, num_points, d_model)*0.02)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 2)
        )

    def forward(self, memory):
        B = memory.size(0)
        q = self.query.expand(B, -1, -1)
        h = self.decoder(q, memory)
        pts = torch.sigmoid(self.head(h))
        return {'points': pts}

# Autoregressive decoder
class AutoregressivePointDecoder(nn.Module):
    def __init__(self, d_model=D_MODEL, num_points=NUM_RADII_64):
        super().__init__()
        self.num_points = num_points
        self.start = nn.Parameter(torch.zeros(1,1,d_model))
        self.gru = nn.GRU(input_size=d_model, hidden_size=d_model, batch_first=True)
        self.ctx = nn.Linear(d_model, d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 2))

    def forward(self, memory):
        B = memory.size(0)
        context = self.ctx(memory.mean(dim=1)).unsqueeze(0)   # [1,B,D]
        inp = self.start.expand(B,1,-1)
        h = context
        outs = []
        for _ in range(self.num_points):
            o, h = self.gru(inp, h)
            xy = torch.sigmoid(self.head(o))  # [B,1,2]
            outs.append(xy)
            # feedback projected to D dims
            pad = torch.zeros(B,1,D_MODEL-2, device=xy.device)
            inp = torch.cat([xy, pad], dim=-1)
        pts = torch.cat(outs, dim=1)
        return {'points': pts}

print('Architectures defined.')

Architectures defined.


## 3) Load checkpoints

In [6]:
def find_checkpoint(name, candidates):
    for p in candidates:
        if p.exists():
            print(f'  {name}: {p}')
            return p
    print(f'  {name}: NOT FOUND')
    return None

polar_ckpt = find_checkpoint('Polar v2', [
    Path('/kaggle/input/datasets/dianisay/trainedmodels/Models/Polar/best.pth'),
    Path('/kaggle/input/polar-v2-weights/checkpoints/best.pth'),
])

detr_ckpt = find_checkpoint('DETR extended', [
    Path('/kaggle/input/datasets/dianisay/trainedmodels/Models/DETR/best.pth'),
    Path('/kaggle/input/detr-extended-weights/checkpoints/best.pth'),
])

ar_ckpt = find_checkpoint('Autoregressive extended', [
    Path('/kaggle/input/datasets/dianisay/trainedmodels/Models/Autoregressive/best.pth'),
    Path('/kaggle/input/autoregressive-extended-weights/checkpoints/best.pth'),
])

  Polar v2: /kaggle/input/datasets/dianisay/trainedmodels/Models/Polar/best.pth
  DETR extended: /kaggle/input/datasets/dianisay/trainedmodels/Models/DETR/best.pth
  Autoregressive extended: /kaggle/input/datasets/dianisay/trainedmodels/Models/Autoregressive/best.pth


In [7]:
models_loaded = []

if polar_ckpt is not None:
    m = PolarModelV2().to(device).eval()
    ck = torch.load(polar_ckpt, map_location=device)
    state = ck['model_state'] if isinstance(ck, dict) and 'model_state' in ck else ck
    m.load_state_dict(state, strict=False)
    models_loaded.append(('Polar v2', m, 'polar_v2'))

if detr_ckpt is not None:
    enc = CNNTransformerEncoder().to(device).eval()
    dec = DETRPointDecoder().to(device).eval()
    ck = torch.load(detr_ckpt, map_location=device)
    if isinstance(ck, dict) and 'encoder_state' in ck and 'decoder_state' in ck:
        enc.load_state_dict(ck['encoder_state'], strict=False)
        dec.load_state_dict(ck['decoder_state'], strict=False)
    elif isinstance(ck, dict) and 'model_state' in ck:
        dec.load_state_dict(ck['model_state'], strict=False)
    else:
        dec.load_state_dict(ck, strict=False)
    models_loaded.append(('DETR extended', (enc, dec), 'detr_extended'))

if ar_ckpt is not None:
    enc = CNNTransformerEncoder().to(device).eval()
    dec = AutoregressivePointDecoder().to(device).eval()
    ck = torch.load(ar_ckpt, map_location=device)
    if isinstance(ck, dict) and 'encoder_state' in ck and 'decoder_state' in ck:
        enc.load_state_dict(ck['encoder_state'], strict=False)
        dec.load_state_dict(ck['decoder_state'], strict=False)
    elif isinstance(ck, dict) and 'model_state' in ck:
        dec.load_state_dict(ck['model_state'], strict=False)
    else:
        dec.load_state_dict(ck, strict=False)
    models_loaded.append(('Autoregressive extended', (enc, dec), 'autoregressive_extended'))

print(f'Loaded models: {[m[0] for m in models_loaded]}')
assert len(models_loaded) > 0, 'No models loaded. Check Kaggle input datasets/paths.'

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 162MB/s] 


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

## 4) Utility functions for differentiation metrics

In [ ]:
def predict_points(model_entry, img_t):
    name, model_or_tuple, mtype = model_entry
    with torch.no_grad():
        if mtype == 'polar_v2':
            out = model_or_tuple(img_t.unsqueeze(0).to(device))
            pts = out['points'][0].detach().cpu().numpy()
        else:
            enc, dec = model_or_tuple
            mem = enc(img_t.unsqueeze(0).to(device))
            out = dec(mem)
            pts = out['points'][0].detach().cpu().numpy()
    pts = np.clip(pts, 0.0, 1.0)
    return pts * IMAGE_SIZE

def polygon_to_mask(pts, size=IMAGE_SIZE):
    m = np.zeros((size, size), dtype=np.uint8)
    pts_i = np.round(pts).astype(np.int32)
    pts_i[:,0] = np.clip(pts_i[:,0], 0, size-1)
    pts_i[:,1] = np.clip(pts_i[:,1], 0, size-1)
    if len(pts_i) >= 3:
        cv2.fillPoly(m, [pts_i], 1)
    return m

def iou_score(pred_mask, gt_mask):
    inter = float((pred_mask & gt_mask).sum())
    union = float((pred_mask | gt_mask).sum())
    return inter/union if union > 0 else 0.0

def dice_score(pred_mask, gt_mask):
    inter = float((pred_mask & gt_mask).sum())
    den = float(pred_mask.sum() + gt_mask.sum())
    return (2*inter)/den if den > 0 else 0.0

def boundary_points(mask):
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(cnts) == 0:
        return np.zeros((0,2), dtype=np.float32)
    c = max(cnts, key=lambda x: x.shape[0]).squeeze()
    if c.ndim == 1:
        return np.zeros((0,2), dtype=np.float32)
    return c.astype(np.float32)

def symmetric_boundary_distance(mask_a, mask_b):
    # Approximate boundary disagreement using distance transforms
    ba = boundary_points(mask_a)
    bb = boundary_points(mask_b)
    if len(ba) == 0 or len(bb) == 0:
        return np.nan

    inv_a = (1 - mask_a).astype(np.uint8)
    inv_b = (1 - mask_b).astype(np.uint8)
    dt_a = cv2.distanceTransform(inv_a, cv2.DIST_L2, 3)
    dt_b = cv2.distanceTransform(inv_b, cv2.DIST_L2, 3)

    ba_i = np.clip(np.round(ba).astype(int), 0, IMAGE_SIZE-1)
    bb_i = np.clip(np.round(bb).astype(int), 0, IMAGE_SIZE-1)
    d_ab = dt_b[ba_i[:,1], ba_i[:,0]].mean()
    d_ba = dt_a[bb_i[:,1], bb_i[:,0]].mean()
    return float((d_ab + d_ba) / 2.0)

def shape_features(mask):
    area = float(mask.sum())
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) == 0:
        return {'area': area, 'perimeter': 0.0, 'compactness': np.nan}
    c = max(cnts, key=cv2.contourArea)
    per = float(cv2.arcLength(c, True))
    comp = (4*np.pi*area)/(per*per + 1e-8) if per > 0 else np.nan
    return {'area': area, 'perimeter': per, 'compactness': comp}

print('Utilities ready.')

## 5) Run all models and compute per-image differentiation signals

In [ ]:
rows = []
pred_store = {}  # (model_name, idx) -> mask

for idx, fimg in enumerate(fixed_images):
    gt = fimg['raw_mask'].astype(np.uint8)
    gt_feat = shape_features(gt)

    for model_entry in models_loaded:
        mname, _, _ = model_entry
        pts = predict_points(model_entry, fimg['image'])
        pred = polygon_to_mask(pts)
        pred_feat = shape_features(pred)

        iou = iou_score(pred, gt)
        dice = dice_score(pred, gt)
        bdist = symmetric_boundary_distance(pred, gt)

        rows.append({
            'image_idx': idx,
            'dataset': fimg['ds_name'],
            'filename': fimg['filename'],
            'model': mname,
            'iou': iou,
            'dice': dice,
            'boundary_dist_px': bdist,
            'pred_area': pred_feat['area'],
            'gt_area': gt_feat['area'],
            'area_error_abs': abs(pred_feat['area'] - gt_feat['area']),
            'pred_perimeter': pred_feat['perimeter'],
            'gt_perimeter': gt_feat['perimeter'],
            'pred_compactness': pred_feat['compactness'],
            'gt_compactness': gt_feat['compactness'],
        })

        pred_store[(mname, idx)] = pred

df = pd.DataFrame(rows)
display(df.head())
print(f'Total records: {len(df)}')

## 6) Differentiation result A: performance distribution by decoder

In [ ]:
summary = df.groupby('model').agg(
    n=('iou','count'),
    iou_mean=('iou','mean'),
    iou_std=('iou','std'),
    dice_mean=('dice','mean'),
    bdist_mean=('boundary_dist_px','mean'),
    area_err_mean=('area_error_abs','mean')
).sort_values('iou_mean', ascending=False)
display(summary)

summary.to_csv('results/differentiation_summary_by_model.csv')
print('Saved: results/differentiation_summary_by_model.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4.5))

for m in df['model'].unique():
    axes[0].hist(df.loc[df.model==m, 'iou'], bins=12, alpha=0.5, label=m)
axes[0].set_title('IoU distribution')
axes[0].set_xlabel('IoU')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

for m in df['model'].unique():
    axes[1].hist(df.loc[df.model==m, 'boundary_dist_px'].dropna(), bins=12, alpha=0.5, label=m)
axes[1].set_title('Boundary disagreement (px)')
axes[1].set_xlabel('Avg symmetric boundary distance')

for m in df['model'].unique():
    axes[2].hist(df.loc[df.model==m, 'area_error_abs'], bins=12, alpha=0.5, label=m)
axes[2].set_title('Absolute area error')
axes[2].set_xlabel('|Pred area - GT area|')

plt.suptitle('Differentiation Signals by Decoder', fontsize=14)
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig('figures/differentiation_metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/differentiation_metric_distributions.png')

## 7) Differentiation result B: per-image model disagreement (what the system is uncertain about)

In [ ]:
model_names = [m[0] for m in models_loaded]

dis_rows = []
for idx, fimg in enumerate(fixed_images):
    masks = [pred_store[(m, idx)] for m in model_names if (m, idx) in pred_store]
    if len(masks) < 2:
        continue

    # Consensus vs disagreement map
    stack = np.stack(masks, axis=0)
    vote_mean = stack.mean(axis=0)                 # [H,W] in [0,1]
    disagreement = vote_mean * (1 - vote_mean)    # max at 0.5

    dis_rows.append({
        'image_idx': idx,
        'dataset': fimg['ds_name'],
        'filename': fimg['filename'],
        'mean_disagreement': float(disagreement.mean()),
        'max_disagreement': float(disagreement.max()),
    })

df_dis = pd.DataFrame(dis_rows).sort_values('mean_disagreement', ascending=False)
display(df_dis.head(10))
df_dis.to_csv('results/per_image_model_disagreement.csv', index=False)
print('Saved: results/per_image_model_disagreement.csv')

In [ ]:
# Visualize top disagreement cases (where differentiation is hardest)
k = min(4, len(df_dis))
if k == 0:
    print('No disagreement results to visualize.')
else:
    top_idx = df_dis.head(k)['image_idx'].tolist()
    fig, axes = plt.subplots(k, 2, figsize=(10, 4*k))
    if k == 1:
        axes = np.array([axes])

    for r, idx in enumerate(top_idx):
        fimg = fixed_images[idx]
        img_show = (fimg['image'] * STD_T + MEAN_T).permute(1,2,0).numpy().clip(0,1)

        masks = [pred_store[(m, idx)] for m in model_names if (m, idx) in pred_store]
        stack = np.stack(masks, axis=0)
        vote_mean = stack.mean(axis=0)
        disagreement = vote_mean * (1 - vote_mean)

        ax0, ax1 = axes[r]
        ax0.imshow(img_show)
        ax0.set_title(f"{fimg['ds_name']} | {fimg['filename']}")
        ax0.axis('off')

        hm = ax1.imshow(disagreement, cmap='magma')
        ax1.set_title('Model disagreement heatmap')
        ax1.axis('off')
        fig.colorbar(hm, ax=ax1, fraction=0.046, pad=0.04)

    plt.suptitle('Top cases where decoders disagree (hard differentiation cases)', fontsize=13)
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig('figures/top_disagreement_cases.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/top_disagreement_cases.png')

## 8) Differentiation result C: simple controller-oriented decision table

In [ ]:
# Build a practical decision support signal from model agreement and expected geometry stability
pivot_iou = df.pivot_table(index='image_idx', columns='model', values='iou', aggfunc='mean')
pivot_area = df.pivot_table(index='image_idx', columns='model', values='pred_area', aggfunc='mean')

controller_rows = []
for idx, fimg in enumerate(fixed_images):
    if idx not in pivot_iou.index:
        continue

iou_vals = pivot_iou.loc[idx].dropna().values
area_vals = pivot_area.loc[idx].dropna().values

iou_mean = float(np.mean(iou_vals)) if len(iou_vals)>0 else np.nan
iou_std  = float(np.std(iou_vals)) if len(iou_vals)>0 else np.nan
area_cv  = float(np.std(area_vals)/(np.mean(area_vals)+1e-8)) if len(area_vals)>0 else np.nan

# Example policy:
# - High confidence: good IoU + low disagreement
# - Medium: moderate metrics
# - Low: poor IoU or high disagreement
if (iou_mean >= 0.75) and (iou_std <= 0.06) and (area_cv <= 0.12):
    confidence = 'HIGH'
    rec_action = 'Proceed autonomous trajectory'
elif (iou_mean >= 0.60) and (iou_std <= 0.12):
    confidence = 'MEDIUM'
    rec_action = 'Proceed with conservative speed + verify boundary'
else:
    confidence = 'LOW'
    rec_action = 'Require human check / fallback strategy'

    controller_rows.append({
        'image_idx': idx,
        'dataset': fimg['ds_name'],
        'filename': fimg['filename'],
        'iou_mean_across_decoders': iou_mean,
        'iou_std_across_decoders': iou_std,
        'pred_area_cv_across_decoders': area_cv,
        'controller_confidence': confidence,
        'recommended_action': rec_action
    })

controller_df = pd.DataFrame(controller_rows)
display(controller_df.head(20))
controller_df.to_csv('results/controller_decision_table.csv', index=False)
print('Saved: results/controller_decision_table.csv')

## 9) Qualitative panel for selected low-confidence cases

In [ ]:
if len(controller_df) == 0:
    print('No controller rows available.')
else:
    sel = controller_df.sort_values('iou_mean_across_decoders', ascending=True).head(min(4, len(controller_df)))
    idxs = sel['image_idx'].tolist()

    fig, axes = plt.subplots(len(idxs), 1, figsize=(10, 4*len(idxs)))
    if len(idxs) == 1:
        axes = [axes]

    color_cycle = ['r', 'b', 'y', 'm']

    for ax, idx in zip(axes, idxs):
        fimg = fixed_images[idx]
        img_show = (fimg['image'] * STD_T + MEAN_T).permute(1,2,0).numpy().clip(0,1)
        ax.imshow(img_show)

        # GT
        cnts, _ = cv2.findContours(fimg['raw_mask'].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cnts:
            p = c.squeeze()
            if p.ndim == 2 and len(p) > 2:
                ax.plot(np.r_[p[:,0], p[0,0]], np.r_[p[:,1], p[0,1]], 'g-', linewidth=2, label='GT')

        # Pred polygons
        for j, model_entry in enumerate(models_loaded):
            mname = model_entry[0]
            pts = predict_points(model_entry, fimg['image'])
            ax.plot(np.r_[pts[:,0], pts[0,0]], np.r_[pts[:,1], pts[0,1]],
                    color=color_cycle[j % len(color_cycle)], linewidth=1.8, label=mname)

        row = controller_df[controller_df.image_idx == idx].iloc[0]
        ax.set_title(
            f"{fimg['ds_name']} | {fimg['filename']} | conf={row['controller_confidence']} | "
            f"IoU mean={row['iou_mean_across_decoders']:.3f}, std={row['iou_std_across_decoders']:.3f}"
        )
        ax.axis('off')

    handles, labels = axes[0].get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    fig.legend(uniq.values(), uniq.keys(), loc='lower center', ncol=min(4, len(uniq)))
    plt.suptitle('Low-confidence cases: GT vs decoder trajectories', fontsize=13, y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    plt.savefig('figures/low_confidence_cases_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/low_confidence_cases_overlay.png')

## 10) Final summary

In [ ]:
print('\n' + '='*70)
print('DIFFERENTIATION ANALYSIS COMPLETE')
print('='*70)
print(f"Models evaluated: {[m[0] for m in models_loaded]}")
print(f"Images analyzed: {len(fixed_images)}")
print(f"Datasets covered: {sorted(set(f['ds_name'] for f in fixed_images))}")
print('\nGenerated CSV outputs:')
print('  results/differentiation_summary_by_model.csv')
print('  results/per_image_model_disagreement.csv')
print('  results/controller_decision_table.csv')
print('\nGenerated figures:')
print('  figures/differentiation_metric_distributions.png')
print('  figures/top_disagreement_cases.png')
print('  figures/low_confidence_cases_overlay.png')
print('\nInterpretation guide:')
print('  - High IoU + low boundary distance => reliable differentiation')
print('  - High cross-decoder disagreement => ambiguous/hard case')
print('  - Controller table translates ML differentiation into action confidence')